在深度学习项目中，数据处理是非常重要的一环，PyTorch提供这两个类来帮助用户高效地加载和处理数据。
- Dataset负责数据的读取和预处理
- DataLoader则负责将数据分成小批量，支持多线程加速，以及数据的打乱等。

### 1. Dataset 模板

Dataset类提供对数据集的抽象，任何自定义数据集都需要继承`torch.utils.data.Dataset`,并实现两个方法：`__len__`和`__getitem__(idx)`。其中`__len__`需要返回整个数据集样本的个数。`__getitem__(idx)`需要能根据样本的index返回具体的样本。

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


class MyDataset(Dataset):
    def __init__(self, X, y=None, dtype=torch.float32):
        """
        X: 可以是 numpy 数组 / pandas.values / list，形状一般 [N, D]
        y: 标签，可选。分类一般是 int64；回归一般是 float32
        """
        self.X = torch.as_tensor(X, dtype=dtype)
        self.y = None if y is None else torch.as_tensor(y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.y is None:
            return x
        return x, self.y[idx]

### 2. 泰坦尼克号数据示例

##### 数据清洗

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)  # 打印时显示所有的列


def load_and_clean_titanic(train_csv_path=None, show_missing=False):
    # 1) 找到 train.csv
    if train_csv_path is None:
        candidates = [
            Path("titanic")
            / "train.csv",  # 推荐：相对路径（从“第七章  - 逻辑回归”目录启动时可用）
            Path(
                r"C:\Users\Anderson\Desktop\RethinkFun深度学习\第七章  - 逻辑回归\titanic\train.csv"
            ),  # 兜底：绝对路径
        ]
        for p in candidates:
            if p.exists():
                train_csv_path = p
                break
        else:
            raise FileNotFoundError("找不到 train.csv，请手动传入 train_csv_path")

    df = pd.read_csv(train_csv_path)

    # 2) 缺失情况（可选展示）
    if show_missing:
        missing_rate = df.isna().mean().sort_values(ascending=False) * 100
        display(missing_rate)
        display(df.isna().sum().sort_values(ascending=False))

    # 3) 删除不必要列
    drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
    df = df.drop(columns=drop_cols)

    # 4) 删除 Age 和 Embarked 缺失的样本
    df = df.dropna(subset=["Age", "Embarked"])

    # 5) 独热编码
    df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True, dtype=int)

    return df


df = load_and_clean_titanic(show_missing=True)
df.head()

Cabin          77.104377
Age            19.865320
Embarked        0.224467
PassengerId     0.000000
Name            0.000000
Pclass          0.000000
Survived        0.000000
Sex             0.000000
Parch           0.000000
SibSp           0.000000
Fare            0.000000
Ticket          0.000000
dtype: float64

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,1,0,1
1,1,1,38.0,1,0,71.2833,0,0,0
2,1,3,26.0,0,0,7.9250,0,0,1
3,1,1,35.0,1,0,53.1000,0,0,1
4,0,3,35.0,0,0,8.0500,1,0,1


##### 自定义数据集，使用数据生成器

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class TabularDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols, label_col=None):
        self.X = torch.tensor(df[feature_cols].values, dtype=torch.float32)

        if label_col is None:
            self.y = None
        else:
            # 二分类标签通常用 long / float 都行：看你的 loss
            # - nn.CrossEntropyLoss  -> y 用 long (0/1/2...)
            # - nn.BCEWithLogitsLoss -> y 用 float (0/1)
            self.y = torch.tensor(df[label_col].values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.y is None:
            return x
        return x, self.y[idx]


# 用法示例
# df = ...  # 你的清洗后的 df
feature_cols = [c for c in df.columns if c != "Survived"]
train_ds = TabularDataset(df, feature_cols=feature_cols, label_col="Survived")

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,  # Windows 上建议先用 0
    drop_last=False,
)

# 取一个 batch 看看
xb, yb = next(iter(train_loader))
xb.shape, yb.shape

(torch.Size([32, 8]), torch.Size([32]))